# 1. T1 플랜과 액션의 일치도

T1을 LIBERO+ Spatial에서 실행합니다. 같은 관측에서 생성한 플랜과 액션 청크를 저장하고,
실제로 실행한 동작을 두 카메라 GIF와 위치·그리퍼 상태로 확인합니다.
이 평가는 T1의 두 출력이 서로 맞는지를 측정합니다.

1. BF16 CUDA GPU 런타임을 선택하고 Colab Secrets에 `HF_TOKEN`을 등록하세요.
2. 소스 셀의 `SOURCE_REF`에 이 평가 코드가 포함된 원격 Git 커밋을 넣으세요.
3. 설정의 `ACTION`을 `plan → setup → run` 순서로 바꾸며 설정 셀과 실행 셀을 실행하세요.
   먼저 `TRIALS=1`, `MAX_STEPS=16`으로 파일럿을 하고, 본 평가에는 새 `RUN_NAME`을 사용하세요.
4. `review.csv`를 복사해 `review-labeled.csv`로 저장하고 각 GIF와 상태를 확인해 아래 기준으로 채점하세요.

| 항목 | 판정 기준 |
|---|---|
| `target_motion` | 플랜이 지정한 물체를 향해 접근하는지, 또는 지정된 방향으로 움직이는지 확인합니다. |
| `gripper` | 지시된 열기·닫기가 실제 그리퍼 상태에 나타나는지 확인합니다. |
| `order` | 플랜이 순서를 명시한 경우 관측된 동작이 그 순서를 따르는지 확인합니다. |

각 항목에 `match`, `mismatch`, `unclear`, `na`를 입력하세요.
관측이 가려졌거나 청크 종료 전에 판정할 수 없으면 `unclear`, 그 항목을 플랜이 언급하지 않았으면 `na`입니다.
빈 셀은 미채점으로 집계합니다. 정해진 간격으로 앞 구간부터 태스크별 최대 8개를 저장하며,
일찍 종료된 태스크는 더 적게 저장됩니다. 열고 닫는 방향은 GIF와 실제 `gripper_qpos`를 보고 판정하세요.


In [ ]:
from pathlib import Path
import os, re, subprocess, sys, time
os.environ.setdefault('VLADE_SESSION_STARTED_UNIX', str(time.time()))
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
sys.dont_write_bytecode = True
SOURCE_REF = 'refs/heads/main'  # 재개는 최초 실행의 SOURCE_COMMIT을 지정합니다.
REPOSITORY = 'https://github.com/eastha10/VLADE.git'
SOURCE_COMMIT = SOURCE_REF if re.fullmatch(r'[0-9a-f]{40}', SOURCE_REF) else subprocess.check_output(['git','ls-remote',REPOSITORY,SOURCE_REF], text=True).split()[0]
if not re.fullmatch(r'[0-9a-f]{40}', SOURCE_COMMIT):
    raise ValueError('불변 Git commit이 필요합니다.')
CODE = Path('/content/vlade_sources') / SOURCE_COMMIT
if any((name == 'vlade' or name.startswith('vlade.')) and getattr(module,'__file__',None) and not Path(module.__file__).resolve().is_relative_to(CODE) for name,module in list(sys.modules.items())):
    raise RuntimeError('다른 소스가 로드되어 있습니다. Colab 런타임을 다시 시작하십시오.')
if not CODE.exists():
    subprocess.run(['git','clone','--no-checkout','--filter=blob:none',REPOSITORY,str(CODE)],check=True)
    subprocess.run(['git','-C',str(CODE),'fetch','--depth','1','origin',SOURCE_COMMIT],check=True)
    subprocess.run(['git','-C',str(CODE),'checkout','--detach',SOURCE_COMMIT],check=True)
sys.path[:0] = [str(CODE/'src'),str(CODE)]
from vlade.utils.source_identity import source_identity
assert source_identity(CODE,require_clean=True)['git_commit']==SOURCE_COMMIT
print('SOURCE_COMMIT:',SOURCE_COMMIT)


In [ ]:
WORKSPACE = "/content/vlade_distillation"
RUN_NAME = "t1-plan-alignment-seed42"
ACTION = "plan"  #@param ["plan", "setup", "run"]
TRIALS = 5
MAX_STEPS = 600
OVERRIDES = dict(trials=TRIALS, max_steps=MAX_STEPS)


In [ ]:
from vlade.legacy.plan_cot.research.plan_evaluation import job

out = job(CODE / 'src/vlade/legacy/plan_cot/configs/experiments/t1_plan_alignment.json', workspace=WORKSPACE,
          run_name=RUN_NAME, action=ACTION, overrides=OVERRIDES, colab=True)


In [ ]:
import json
from IPython.display import JSON, display
from vlade.legacy.plan_cot.research.plan_evaluation import output_path

out = output_path(Path(WORKSPACE), RUN_NAME)
if (out / "results.json").exists():
    display(JSON(json.loads((out / "results.json").read_text(encoding="utf-8"))))
else:
    print("결과 파일은 run 단계가 완료되면 생성됩니다.")


In [ ]:
from IPython.display import Image, display

CLIP_ID = ""  # review.csv에서 확인할 chunk_id 입력
if CLIP_ID:
    display(Image(filename=str(out / "clips" / (CLIP_ID + ".gif"))))


In [ ]:
from vlade.legacy.plan_cot.evaluation.plan_rollout import summarize_alignment

scored = out / "review-labeled.csv"
if scored.exists():
    display(JSON(summarize_alignment(scored)))
